# Download the dataset and upload it to colab from this [link](https://www.kaggle.com/datasets/riotulab/saudi-license-plate-characters/data)

##### Note that colab was used in development so minor changes might be required

In [ ]:
!unzip -q ../data/archive.zip -d ../data/saudi-license-plate-characters

In [ ]:
!ls ../data/saudi-license-plate-characters

License-Characters-by-2-27classes


In [ ]:
import os
import shutil
import random

# Set your paths
SOURCE_TRAIN = "../data/saudi-license-plate-characters/License-Characters-by-2-27classes/train"
SOURCE_TEST  = "../data/saudi-license-plate-characters/License-Characters-by-2-27classes/test"
DEST         = "dataset"
VAL_SPLIT    = 0.10  # 10% of train → val

# Create destination folders
for split in ["train", "val", "test"]:
    os.makedirs(f"{DEST}/images/{split}", exist_ok=True)
    os.makedirs(f"{DEST}/labels/{split}", exist_ok=True)

def copy_files(src_dir, split):
    for fname in os.listdir(src_dir):
        src = os.path.join(src_dir, fname)
        if fname.endswith((".jpeg", ".jpg", ".png")):
            shutil.copy(src, f"{DEST}/images/{split}/{fname}")
        elif fname.endswith(".txt"):
            shutil.copy(src, f"{DEST}/labels/{split}/{fname}")


# Handle test set
copy_files(SOURCE_TEST, "test")

# split train into train/val
all_images = [f for f in os.listdir(SOURCE_TRAIN)
              if f.endswith((".jpeg", ".jpg", ".png"))]

random.seed(42)
random.shuffle(all_images)

val_count   = int(len(all_images) * VAL_SPLIT)
val_images  = set(all_images[:val_count])
train_images = set(all_images[val_count:])

for fname in os.listdir(SOURCE_TRAIN):
    src = os.path.join(SOURCE_TRAIN, fname)
    stem = os.path.splitext(fname)[0]  # filename without extension

    if fname.endswith((".jpeg", ".jpg", ".png")):
        split = "val" if fname in val_images else "train"
        shutil.copy(src, f"{DEST}/images/{split}/{fname}")
    elif fname.endswith((".txt", ".xml")):
        # Match label to its image's split
        img_name = next((img for img in all_images if os.path.splitext(img)[0] == stem), None)
        if img_name:
            split = "val" if img_name in val_images else "train"
            shutil.copy(src, f"{DEST}/labels/{split}/{fname}")

print(f"Train images: {len(os.listdir(f'{DEST}/images/train'))}")
print(f"Val images:   {len(os.listdir(f'{DEST}/images/val'))}")
print(f"Test images:  {len(os.listdir(f'{DEST}/images/test'))}")

Train images: 507
Val images:   56
Test images:  30


In [ ]:
import cv2

try:
    from google.colab.patches import cv2_imshow
except ImportError:
    def cv2_imshow(img):
        cv2.imshow("Image", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()

def visualize(images_dir, labels_dir):
    # pick random image
    img_file = random.choice(os.listdir(images_dir))
    img_path = os.path.join(images_dir, img_file)
    lbl_path = os.path.join(labels_dir, os.path.splitext(img_file)[0] + ".txt")

    img = cv2.imread(img_path)
    h, w = img.shape[:2]

    with open(lbl_path) as f:
        for line in f:
            cls, xc, yc, bw, bh = map(float, line.strip().split())
            x1 = int((xc - bw/2) * w)
            y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w)
            y2 = int((yc + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, str(int(cls)), (x1, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    cv2_imshow(img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

visualize(f"{DEST}/images/train", f"{DEST}/labels/train")

In [5]:
# Extract label names from xml files to create .yaml file required for YOLO
import xml.etree.ElementTree as ET
from collections import defaultdict

xml_dir = "dataset/labels/train"
txt_dir = "dataset/labels/train"

# for each file, map class_id from txt to class_name from xml
class_map = {}

for fname in os.listdir(xml_dir):
    if fname.endswith(".xml"):
        stem = os.path.splitext(fname)[0]
        txt_path = os.path.join(txt_dir, stem + ".txt")
        xml_path = os.path.join(xml_dir, fname)

        if not os.path.exists(txt_path):
            continue

        # get names from xml in order
        tree = ET.parse(xml_path)
        names = [obj.find("name").text for obj in tree.getroot().findall("object")]

        # get class ids from txt in order
        with open(txt_path) as f:
            ids = [int(line.strip().split()[0]) for line in f]

        for cls_id, name in zip(ids, names):
            class_map[cls_id] = name

for cls_id in sorted(class_map.keys()):
    print(f"{cls_id}: {class_map[cls_id]}")

0: 0
1: 1
2: 2
3: 3
4: 4
5: 5
6: 6
7: 7
8: 8
9: 9
10: A
11: B
12: D
13: E
14: G
15: H
16: J
17: K
18: L
19: N
20: R
21: S
22: T
23: U
24: V
25: X
26: Z


In [6]:
import yaml

class_map = {
    0: "0", 1: "1", 2: "2", 3: "3", 4: "4",
    5: "5", 6: "6", 7: "7", 8: "8", 9: "9",
    10: "A", 11: "B", 12: "D", 13: "E", 14: "G",
    15: "H", 16: "J", 17: "K", 18: "L", 19: "N",
    20: "R", 21: "S", 22: "T", 23: "U", 24: "V",
    25: "X", 26: "Z"
}

data = {
    "path": "dataset",
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": 27,
    "names": class_map
}

with open("dataset/data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=True, allow_unicode=True)

print("data.yaml created successfully")

data.yaml created successfully


In [7]:
with open("dataset/data.yaml") as f:
    print(f.read())

names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'
  10: A
  11: B
  12: D
  13: E
  14: G
  15: H
  16: J
  17: K
  18: L
  19: N
  20: R
  21: S
  22: T
  23: U
  24: V
  25: X
  26: Z
nc: 27
path: dataset
test: images/test
train: images/train
val: images/val



In [8]:
! pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.5 MB/s eta 0:00:00


In [ ]:
import albumentations as A
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

augmentations = [
    A.BBoxSafeRandomCrop(p=0.5),
    A.Resize(640, 640),
    A.RandomBrightnessContrast(p=0.2),
    A.HorizontalFlip(p=0.5),
    A.OneOf([
        A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.1, 0.25),
                        hole_width_range=(0.1, 0.25), p=1.0),
        A.GridDropout(ratio=0.5, unit_size_range=(10, 20), p=1.0)
    ], p=0.5)
]

results = model.train(
    data=f"{DEST}/data.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    augmentations=augmentations
)

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")

# pick a random test image
test_dir = f"{DEST}/images/test"
img_file = random.choice(os.listdir(test_dir))
img_path = os.path.join(test_dir, img_file)

results = model.predict(img_path, conf=0.616)  # use the best confidence threshold
results[0].show()  # displays the image with boxes and class labels

In [ ]:
WEIGHTS_DIR = "runs/detect/train/weights"

model.export(format='onnx')

try:
    from google.colab import files
    files.download(f"{WEIGHTS_DIR}/best.pt")
    files.download(f"{WEIGHTS_DIR}/best.onnx")
    files.download(f"{WEIGHTS_DIR}/last.pt")
    files.download("runs/detect/train/results.csv")
except ImportError:
    print("Not running in Colab. Files are saved locally at runs/detect/train/")